# P1-E16.7F Renamed(P1-E11.F) Four-Endpoint Fold-Safe Age Confounding and Age-Normative Sensitivity Audit

## Purpose
Determine, under the exact final P1-E16.7A/P1-E16.7B four-endpoint outer-fold contract, how much diagnostic information is carried by age alone, whether EEG adds information beyond age, whether adding age improves EEG classification, and whether regional EEG conclusions remain stable after strictly fold-fitted age normalization.

This is a confounding and robustness analysis, not a performance-optimization experiment. Official-test recordings and external cohorts remain excluded.


## Analysis contract

- Endpoints: Normal vs AD, MCI vs AD, Normal vs MCI, and Normal/MCI/AD three-class classification.
- Outer-fold membership is imported from the authenticated P1-E16.7A split audit used by P1-E16.7B and cannot be regenerated.
- Every transform, normative regression, feature selection, classifier choice, and binary threshold is fitted using outer-training data only.
- The Normal normative pool excludes every recording or identity group held out in the corresponding outer fold, including endpoints that do not contain Normal.
- All representations use the same age-complete recordings within an endpoint.
- Paired bootstrap intervals and Holm-adjusted comparisons are computed across the prespecified age contrasts.
- No performance target is used as a selection or reporting gate.


In [ ]:
# 1. Configuration and imports
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, re, sys, time, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (balanced_accuracy_score, f1_score,
    matthews_corrcoef, roc_auc_score, confusion_matrix)
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

warnings.filterwarnings('ignore', category=FutureWarning)

EXPERIMENT_ROOT = Path(r'D:\EEG_AD_Research\dementia\paper one')
OUTPUT_ROOT = EXPERIMENT_ROOT/'paper1_outputs'/'p1_e16_7f_four_endpoint_age_confounding_normative_sensitivity'
TABLES = OUTPUT_ROOT/'tables'; FIGURES = OUTPUT_ROOT/'figures'; CACHE = OUTPUT_ROOT/'job_cache'
for folder in (OUTPUT_ROOT, TABLES, FIGURES, CACHE): folder.mkdir(parents=True, exist_ok=True)

FEATURE_NPZ = None          # set an exact Path only if discovery reports duplicates
METADATA_CSV = None         # set an exact Path only if discovery reports duplicates
AGE_COLUMN = None           # set explicitly only if conservative detection is ambiguous
RUN_PROFILE = 'full'
BASE_SEED = 20260914
TASKS = {'normal_vs_ad':['Normal','AD'], 'mci_vs_ad':['MCI','AD'],
         'normal_vs_mci':['Normal','MCI'], 'three_class':['Normal','MCI','AD']}
OUTER_FOLDS, INNER_FOLDS, N_BOOT = 5, 3, 2000
CANONICAL_CANDIDATES = [
    EXPERIMENT_ROOT/'paper1_outputs'/'p1_e16_7a_four_endpoint_nested_conventional_freeze'/'tables'/'07_OUTER_SPLIT_AUDIT.csv']

FEATURE_CAP = 120
MIN_NORMAL_REFERENCE = 20
EPS = 1e-8
np.random.seed(BASE_SEED)

print('Final four-endpoint fold-safe age sensitivity analysis')
print('Output:', OUTPUT_ROOT)

In [ ]:
# 2. Discover, authenticate, and load locked P1-E08.1 inputs
FEATURE_FILENAME='02_LOCKED_RECORDING_LEVEL_NONSPIKING_FEATURE_MATRIX.npz'
METADATA_FILENAME='05_LOCKED_RECORDING_FEATURE_METADATA_WITH_LABELS.csv'

def sha256_file(path, block=2**20):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(block), b''): h.update(b)
    return h.hexdigest()

def discover_exact(filename):
    paths=[]
    for p in EXPERIMENT_ROOT.rglob(filename):
        parts={x.lower() for x in p.parts}
        if '.ipynb_checkpoints' in parts or any(x.startswith('p1_e13') for x in parts): continue
        paths.append(p.resolve())
    return sorted(set(paths), key=lambda p:str(p).lower())

feature_candidates=[Path(FEATURE_NPZ).resolve()] if FEATURE_NPZ else discover_exact(FEATURE_FILENAME)
metadata_candidates=[Path(METADATA_CSV).resolve()] if METADATA_CSV else discover_exact(METADATA_FILENAME)
inventory=pd.DataFrame(
    [{'artifact':'feature_npz','path':str(p),'exists':p.is_file()} for p in feature_candidates]+
    [{'artifact':'metadata_csv','path':str(p),'exists':p.is_file()} for p in metadata_candidates])
display(inventory)
if len(feature_candidates)!=1: raise RuntimeError(f'Expected one locked feature matrix; found {len(feature_candidates)}. Set FEATURE_NPZ exactly.')
if len(metadata_candidates)!=1: raise RuntimeError(f'Expected one locked metadata table; found {len(metadata_candidates)}. Set METADATA_CSV exactly.')
FEATURE_NPZ, METADATA_CSV = feature_candidates[0], metadata_candidates[0]

with np.load(FEATURE_NPZ,allow_pickle=False) as z:
    required={'X','recording_id','feature_names','representation_group'}
    if not required.issubset(z.files): raise KeyError(f'NPZ missing: {sorted(required-set(z.files))}')
    X_all=np.asarray(z['X'],dtype=np.float32)
    recording_ids=np.asarray([str(x).strip().zfill(5) for x in z['recording_id'].astype(str)])
    feature_names=z['feature_names'].astype(str)
    representation_groups=z['representation_group'].astype(str)

metadata=pd.read_csv(METADATA_CSV,dtype={'recording_id':str})
metadata['recording_id']=metadata.recording_id.astype(str).str.strip().str.zfill(5)
required_meta={'recording_id','official_split','experimental_label'}
if not required_meta.issubset(metadata): raise KeyError(f'Metadata missing: {sorted(required_meta-set(metadata))}')
if X_all.shape != (len(metadata),len(feature_names)): raise ValueError('Feature/metadata/name shape mismatch.')
if not np.array_equal(recording_ids,metadata.recording_id.astype(str).to_numpy()): raise ValueError('Feature rows and metadata IDs are not identically ordered.')
if not np.isfinite(X_all).all(): raise ValueError('Locked feature matrix contains non-finite values.')

development_mask=metadata.official_split.astype(str).str.lower().isin(['train','validation']).to_numpy()
X=X_all[development_mask]
dev=metadata.loc[development_mask].reset_index(drop=True)
ids=recording_ids[development_mask]
labels=dev.experimental_label.astype(str).to_numpy()
groups=(dev.identity_group.astype(str).to_numpy() if 'identity_group' in dev else ids.copy())
if dev.official_split.astype(str).str.lower().eq('test').any(): raise AssertionError('STOP: official test entered P1-E16.7F.')

input_inventory=pd.DataFrame([
 {'artifact':'feature_npz','path':str(FEATURE_NPZ),'bytes':FEATURE_NPZ.stat().st_size,'sha256':sha256_file(FEATURE_NPZ)},
 {'artifact':'metadata_csv','path':str(METADATA_CSV),'bytes':METADATA_CSV.stat().st_size,'sha256':sha256_file(METADATA_CSV)}])
input_inventory.to_csv(TABLES/'01_LOCKED_INPUT_INVENTORY.csv',index=False)
display(input_inventory)
print('Development:',X.shape,pd.Series(labels).value_counts().to_dict())
print('Official test accessed: False')

# Authenticate the canonical development-fold contract.
canonical_path=next((p for p in CANONICAL_CANDIDATES if p.is_file()),None)
if canonical_path is None:
    matches=sorted(EXPERIMENT_ROOT.rglob('*OOF*PREDICTIONS*.csv'))
    canonical_path=next((p for p in matches if 'e16_7a' in str(p).lower() and 'outer_split_audit' in p.name.lower()),None)
if canonical_path is None:raise FileNotFoundError('Canonical P1-E16.1 OOF prediction file was not found.')
canonical=pd.read_csv(canonical_path,dtype={'recording_id':str})
canonical.columns=[str(c).strip().lower() for c in canonical.columns]
canonical=canonical.rename(columns={'endpoint':'task','fold':'outer_fold'})
if not {'task','outer_fold','recording_id'}.issubset(canonical):raise KeyError('Canonical file requires task, outer_fold, recording_id.')
if 'role' in canonical.columns:
    canonical=canonical.loc[canonical.role.astype(str).str.lower().isin(['outer_holdout','outer_test','test'])].copy()
canonical['recording_id']=canonical.recording_id.astype(str).str.strip().str.zfill(5)
canonical['outer_fold']=pd.to_numeric(canonical.outer_fold,errors='raise').astype(int)
canonical=canonical.loc[canonical.task.isin(TASKS),['task','outer_fold','recording_id']].drop_duplicates(['task','recording_id'])


In [ ]:
# 3. Resolve age and audit the analysis cohort
def normalized(s): return re.sub(r'[^a-z0-9]+','_',str(s).lower()).strip('_')
exact_age={'age','age_years','age_at_recording','age_at_exam','patient_age','subject_age'}
age_candidates=[c for c in dev.columns if normalized(c) in exact_age]
if AGE_COLUMN is None:
    if len(age_candidates)!=1:
        raise RuntimeError(f'Expected exactly one conservative age candidate; found {age_candidates}. Set AGE_COLUMN explicitly after inspection.')
    AGE_COLUMN=age_candidates[0]
if AGE_COLUMN not in dev: raise KeyError(f'AGE_COLUMN={AGE_COLUMN!r} not found.')
ages=pd.to_numeric(dev[AGE_COLUMN],errors='coerce').to_numpy(float)

age_audit=(pd.DataFrame({'label':labels,'age':ages})
           .groupby('label',as_index=False).agg(n=('age','size'),n_age=('age','count'),
            missing_age=('age',lambda x:int(x.isna().sum())),age_mean=('age','mean'),
            age_sd=('age','std'),age_min=('age','min'),age_median=('age','median'),age_max=('age','max')))
age_audit.to_csv(TABLES/'02_AGE_AUDIT.csv',index=False)
display(age_audit.round(2))

# A fair complete-case cohort is used for every representation so age-normative
# and raw comparisons have identical observations.
keep=np.isfinite(ages) & np.isin(labels,['Normal','MCI','AD'])
coverage=float(keep.mean())
if coverage<0.70: raise ValueError(f'Age-complete coverage is only {coverage:.1%}; review the age field before modelling.')
X,dev,ids,labels,groups,ages = X[keep],dev.loc[keep].reset_index(drop=True),ids[keep],labels[keep],groups[keep],ages[keep]
if pd.Series(ids).duplicated().any(): raise ValueError('Duplicate recording_id rows detected.')
print(f'Complete-case cohort: {len(ids)}/{len(keep)} ({coverage:.1%})')

# The canonical folds must exactly cover the age-complete cohort.
fold_checks=[]
for task,classes in TASKS.items():
    expected=set(ids[np.isin(labels,classes)]);q=canonical.loc[canonical.task.eq(task)]
    observed=set(q.recording_id)
    fold_checks.append({'task':task,'expected_n':len(expected),'assigned_n':len(observed),
                        'exact_coverage':expected==observed,'n_folds':q.outer_fold.nunique()})
fold_contract=pd.DataFrame(fold_checks);display(fold_contract)
if not fold_contract.exact_coverage.all() or not fold_contract.n_folds.eq(OUTER_FOLDS).all():
    raise AssertionError('Age-complete cohort does not match the canonical fold contract.')
canonical.to_csv(TABLES/'03_CANONICAL_OUTER_FOLD_MANIFEST.csv',index=False)
fold_contract.to_csv(TABLES/'04_CANONICAL_FOLD_CONTRACT.csv',index=False)
print('Canonical fold source:',canonical_path)


## Data

### Regional representation

The mapping below is anatomical and label-independent. Each channel-resolved feature is assigned to frontal, central, temporal, parietal, or occipital cortex. Within every region and feature stem, the notebook calculates the channel mean and dispersion. Features without a recognized channel token are excluded from the regional view but remain available in the raw hybrid comparator.

In [ ]:
# 4. Build fixed channel-to-region aggregation
REGIONS={
 'frontal':['fp1','fp2','f3','f4','f7','f8','fz'],
 'central':['c3','c4','cz'],
 'temporal':['t3','t4','t5','t6','t7','t8'],
 'parietal':['p3','p4','p7','p8','pz'],
 'occipital':['o1','o2','oz']}
CHANNEL_TO_REGION={ch:r for r,chs in REGIONS.items() for ch in chs}
channel_pattern=re.compile(r'(^|_)('+'|'.join(sorted(CHANNEL_TO_REGION,key=len,reverse=True))+r')($|_)')

def parse_feature(name):
    s=normalized(name); m=channel_pattern.search(s)
    if not m: return None,None,None
    ch=m.group(2); region=CHANNEL_TO_REGION[ch]
    stem=re.sub(r'(^|_)'+re.escape(ch)+r'($|_)','_',s)
    stem=re.sub(r'_+','_',stem).strip('_') or 'value'
    return ch,region,stem

parsed=[parse_feature(n) for n in feature_names]
mapping=pd.DataFrame({'feature_index':np.arange(len(feature_names)),'feature_name':feature_names,
 'representation_group':representation_groups,'channel':[x[0] for x in parsed],
 'region':[x[1] for x in parsed],'feature_stem':[x[2] for x in parsed]})
mapping.to_csv(TABLES/'03_FEATURE_TO_REGION_MAP.csv',index=False)
display(mapping.region.value_counts(dropna=False).rename_axis('region').reset_index(name='n_features'))

regional_blocks=[]; regional_names=[]
for (rep,region,stem),q in mapping.dropna(subset=['region']).groupby(['representation_group','region','feature_stem'],sort=True):
    idx=q.feature_index.to_numpy(int)
    if len(idx)<2: continue
    block=X[:,idx]
    regional_blocks.extend([block.mean(axis=1),block.std(axis=1,ddof=0)])
    regional_names.extend([f'{rep}__{region}__{stem}__mean',f'{rep}__{region}__{stem}__sd'])
if not regional_blocks: raise ValueError('No multi-channel regional features could be constructed; inspect 03_FEATURE_TO_REGION_MAP.csv.')
X_regional=np.column_stack(regional_blocks).astype(np.float32)
regional_names=np.asarray(regional_names)
if not np.isfinite(X_regional).all(): raise ValueError('Regional matrix contains non-finite values.')
print('Raw hybrid:',X.shape,'| regional:',X_regional.shape)
display(pd.Series([n.split('__')[1] for n in regional_names]).value_counts().rename_axis('region').reset_index(name='n_features'))

In [ ]:
# 5. Fold-safe transforms, classifiers, and metrics
REPRESENTATIONS=['age_only','hybrid_raw','hybrid_raw_plus_age','regional_raw','regional_raw_plus_age','regional_age_normative','regional_combined']

def age_normative(train_idx,test_idx,normal_ref_idx):
    ref=np.asarray(normal_ref_idx,int)
    if len(ref)<MIN_NORMAL_REFERENCE: raise ValueError(f'Too few Normal references: {len(ref)}')
    if not np.all(labels[ref]=='Normal'): raise AssertionError('Normative pool contains non-Normal rows.')
    model=Ridge(alpha=1.0).fit(np.c_[ages[ref],ages[ref]**2],X_regional[ref])
    ref_res=X_regional[ref]-model.predict(np.c_[ages[ref],ages[ref]**2])
    center=np.median(ref_res,axis=0)
    scale=1.4826*np.median(np.abs(ref_res-center),axis=0)
    good=np.isfinite(center)&np.isfinite(scale)&(scale>1e-7)
    if good.sum()<2: raise ValueError('No stable regional normative features.')
    def z(ii):
        residual=X_regional[ii]-model.predict(np.c_[ages[ii],ages[ii]**2])
        return np.clip(np.nan_to_num((residual[:,good]-center[good])/scale[good],nan=0,posinf=8,neginf=-8),-8,8)
    return z(train_idx),z(test_idx),good

def transform_rep(rep,train_idx,test_idx,normal_ref_idx):
    if rep=='age_only':
        mu=ages[train_idx].mean(); sd=ages[train_idx].std(ddof=1)+EPS
        return ((ages[train_idx]-mu)/sd)[:,None],((ages[test_idx]-mu)/sd)[:,None]
    age_mu=ages[train_idx].mean();age_sd=ages[train_idx].std(ddof=1)+EPS
    age_tr=((ages[train_idx]-age_mu)/age_sd)[:,None];age_te=((ages[test_idx]-age_mu)/age_sd)[:,None]
    if rep=='hybrid_raw': return X[train_idx],X[test_idx]
    if rep=='hybrid_raw_plus_age': return np.c_[X[train_idx],age_tr],np.c_[X[test_idx],age_te]
    if rep=='regional_raw': return X_regional[train_idx],X_regional[test_idx]
    if rep=='regional_raw_plus_age': return np.c_[X_regional[train_idx],age_tr],np.c_[X_regional[test_idx],age_te]
    zn_tr,zn_te,_=age_normative(train_idx,test_idx,normal_ref_idx)
    if rep=='regional_age_normative': return zn_tr,zn_te
    if rep=='regional_combined': return np.c_[X_regional[train_idx],zn_tr],np.c_[X_regional[test_idx],zn_te]
    raise KeyError(rep)

def make_pipeline(spec,seed):
    name,param=spec
    if name=='logreg':
        model=LogisticRegression(C=param,class_weight='balanced',solver='lbfgs',max_iter=5000,random_state=seed)
        return Pipeline([('scale',StandardScaler()),('select',SelectKBest(f_classif,k=FEATURE_CAP)),('model',model)])
    if name=='extra_trees':
        model=ExtraTreesClassifier(n_estimators=350,max_features=param,min_samples_leaf=2,
            class_weight='balanced',n_jobs=-1,random_state=seed)
        return Pipeline([('select',SelectKBest(f_classif,k=FEATURE_CAP)),('model',model)])
    raise KeyError(spec)

CANDIDATES=[('logreg',0.03),('logreg',0.1),('logreg',0.3),('logreg',1.0),
            ('extra_trees','sqrt'),('extra_trees',0.35),('extra_trees',0.6)]

def safe_pipeline(spec,seed,n_features):
    pipe=make_pipeline(spec,seed)
    pipe.set_params(select__k=min(FEATURE_CAP,n_features))
    return pipe

def aligned_proba(model,x,class_order):
    p=model.predict_proba(x); out=np.zeros((len(x),len(class_order)))
    for j,c in enumerate(model.classes_): out[:,class_order.index(str(c))]=p[:,j]
    out=np.clip(out,1e-7,1-1e-7); return out/out.sum(axis=1,keepdims=True)

def metrics(y_true,y_pred,proba,class_order):
    out={'balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
         'macro_f1':f1_score(y_true,y_pred,average='macro'),
         'mcc':matthews_corrcoef(y_true,y_pred)}
    try:
        if len(class_order)==2:
            yy=(np.asarray(y_true)==class_order[1]).astype(int)
            out['roc_auc']=roc_auc_score(yy,proba[:,1])
        else:
            y_bin=label_binarize(y_true,classes=class_order)
            out['roc_auc']=roc_auc_score(y_bin,proba,multi_class='ovr',average='macro')
    except ValueError: out['roc_auc']=np.nan
    return out

def best_threshold(y_true,p_positive,positive_label):
    grid=np.linspace(.05,.95,181); yy=(np.asarray(y_true)==positive_label).astype(int)
    scores=np.asarray([balanced_accuracy_score(yy,(p_positive>=t).astype(int)) for t in grid])
    best=np.flatnonzero(scores==scores.max()); return float(grid[best[np.argmin(np.abs(grid[best]-.5))]])

In [ ]:
# 6. One fully fold-safe outer job (with fold-safe inner selection and thresholding)
def normal_reference(task_classes,diagnostic_train_idx,permitted_all_idx,prohibited_idx):
    diagnostic_train_idx=np.asarray(diagnostic_train_idx,int); permitted_all_idx=np.asarray(permitted_all_idx,int)
    if 'Normal' in task_classes: ref=diagnostic_train_idx[labels[diagnostic_train_idx]=='Normal']
    else: ref=permitted_all_idx[labels[permitted_all_idx]=='Normal']
    ref=np.unique(ref)
    if len(ref)<MIN_NORMAL_REFERENCE: raise ValueError(f'Insufficient Normal reference: {len(ref)}')
    if np.intersect1d(ref,prohibited_idx).size: raise AssertionError('Held-out rows leaked into normative reference.')
    if not np.all(np.isin(ref,permitted_all_idx)): raise AssertionError('Normative pool escaped permitted training data.')
    return ref

def cache_path(task,rep,seed,fold):
    return CACHE/f'{task}__{rep}__seed{seed}__fold{fold}__final_four_endpoint_v1.npz'

def run_outer_job(task,task_classes,rep,seed,fold_no,tr_pos,te_pos,task_idx):
    cp=cache_path(task,rep,seed,fold_no)
    if cp.exists():
        with np.load(cp,allow_pickle=False) as z:
            return json.loads(str(z['record'])),pd.DataFrame(json.loads(str(z['predictions'])))
    tr=task_idx[tr_pos]; te=task_idx[te_pos]
    ytr,yte=labels[tr],labels[te]
    # Exclude the complete canonical fold, not merely endpoint test rows. This
    # prevents held-out Normal recordings entering the MCI-vs-AD normative pool.
    fold_holdout_ids=set(canonical.loc[(canonical.task=='three_class') & (canonical.outer_fold==int(fold_no)),'recording_id'])
    prohibited_idx=np.flatnonzero(np.isin(ids,list(fold_holdout_ids)))
    prohibited_groups=np.unique(groups[prohibited_idx])
    permitted_all=np.flatnonzero(~np.isin(groups,prohibited_groups))
    if np.intersect1d(groups[permitted_all],prohibited_groups).size: raise AssertionError('Group leakage in outer split.')

    inner=StratifiedGroupKFold(n_splits=INNER_FOLDS,shuffle=True,random_state=seed+fold_no)
    inner_splits=list(inner.split(np.zeros(len(tr)),ytr,groups[tr]))
    candidate_scores={spec:[] for spec in CANDIDATES}
    for itr_pos,iva_pos in inner_splits:
        itr,iva=tr[itr_pos],tr[iva_pos]
        inner_prohibited=np.unique(groups[iva])
        inner_permitted=permitted_all[~np.isin(groups[permitted_all],inner_prohibited)]
        ref=normal_reference(task_classes,itr,inner_permitted,iva)
        xa,xb=transform_rep(rep,itr,iva,ref)
        for spec in CANDIDATES:
            model=safe_pipeline(spec,seed+fold_no,xa.shape[1]).fit(xa,labels[itr])
            candidate_scores[spec].append(balanced_accuracy_score(labels[iva],model.predict(xb)))
    means={k:float(np.mean(v)) for k,v in candidate_scores.items()}
    selected=max(CANDIDATES,key=lambda k:(means[k],-CANDIDATES.index(k)))

    # Generate strictly inner-held-out probabilities for threshold selection.
    inner_truth=[]; inner_prob=[]
    if len(task_classes)==2:
        for itr_pos,iva_pos in inner_splits:
            itr,iva=tr[itr_pos],tr[iva_pos]
            inner_prohibited=np.unique(groups[iva])
            inner_permitted=permitted_all[~np.isin(groups[permitted_all],inner_prohibited)]
            ref=normal_reference(task_classes,itr,inner_permitted,iva)
            xa,xb=transform_rep(rep,itr,iva,ref)
            model=safe_pipeline(selected,seed+fold_no,xa.shape[1]).fit(xa,labels[itr])
            inner_truth.extend(labels[iva]); inner_prob.extend(aligned_proba(model,xb,task_classes)[:,1])
        threshold=best_threshold(inner_truth,np.asarray(inner_prob),task_classes[1])
    else: threshold=np.nan

    ref=normal_reference(task_classes,tr,permitted_all,te)
    xa,xb=transform_rep(rep,tr,te,ref)
    model=safe_pipeline(selected,seed+fold_no,xa.shape[1]).fit(xa,ytr)
    proba=aligned_proba(model,xb,task_classes)
    if len(task_classes)==2: pred=np.where(proba[:,1]>=threshold,task_classes[1],task_classes[0])
    else: pred=np.asarray(task_classes)[proba.argmax(axis=1)]
    met=metrics(yte,pred,proba,task_classes)
    rec={'task':task,'representation':rep,'seed':seed,'outer_fold':fold_no,
         'n_train':len(tr),'n_test':len(te),'normal_reference_n':len(ref),
         'selected_model':selected[0],'selected_parameter':str(selected[1]),
         'inner_balanced_accuracy':means[selected],'threshold':threshold,**met}
    pp=pd.DataFrame({'task':task,'representation':rep,'seed':seed,'outer_fold':fold_no,
        'recording_id':ids[te],'identity_group':groups[te],'age':ages[te],
        'true_label':yte,'predicted_label':pred})
    for j,c in enumerate(task_classes): pp[f'probability_{c}']=proba[:,j]
    np.savez_compressed(cp,record=np.asarray(json.dumps(rec)),predictions=np.asarray(pp.to_json(orient='records')))
    return rec,pp

## Results

### Repeated recording-disjoint evaluation

The following cell is resumable. In the full profile it evaluates 3 endpoints x 5 representations x 3 seeds x 5 folds = 225 outer jobs. Each job contains its own inner model selection and, for binary tasks, inner-held-out threshold estimation.


In [ ]:
# 7. Run or resume the canonical outer-fold jobs
records=[];prediction_parts=[];seed=BASE_SEED
total=len(TASKS)*len(REPRESENTATIONS)*OUTER_FOLDS;done=0;started=time.time()
for task,classes in TASKS.items():
    task_idx=np.flatnonzero(np.isin(labels,classes));task_ids=ids[task_idx]
    task_folds=canonical.loc[canonical.task.eq(task),['recording_id','outer_fold']]
    for rep in REPRESENTATIONS:
        for fold_value in sorted(task_folds.outer_fold.unique()):
            fold_no=int(fold_value);test_ids=set(task_folds.loc[task_folds.outer_fold.eq(fold_no),'recording_id'])
            te_pos=np.flatnonzero(np.isin(task_ids,list(test_ids)));tr_pos=np.flatnonzero(~np.isin(task_ids,list(test_ids)))
            if set(task_ids[te_pos])!=test_ids:raise AssertionError(f'{task} fold {fold_no}: test membership mismatch.')
            rec,pp=run_outer_job(task,classes,rep,seed,fold_no,tr_pos,te_pos,task_idx)
            records.append(rec);prediction_parts.append(pp);done+=1
            print(f'[{done:03d}/{total}] {task:12s} {rep:26s} fold={fold_no} BA={rec["balanced_accuracy"]:.3f}')
fold_results=pd.DataFrame(records);oof=pd.concat(prediction_parts,ignore_index=True)
fold_results.to_csv(TABLES/'05_OUTER_FOLD_RESULTS.csv',index=False);oof.to_csv(TABLES/'06_OOF_PREDICTIONS.csv',index=False)
print(f'Completed {done} jobs in {(time.time()-started)/60:.1f} minutes')
assert done==total


In [ ]:
# 8. Integrity checks
checks=[]
for task,classes in TASKS.items():
    expected=set(ids[np.isin(labels,classes)]);probability_cols=[f'probability_{c}' for c in classes]
    expected_fold=canonical.loc[canonical.task.eq(task)].set_index('recording_id').outer_fold.astype(int)
    for rep in REPRESENTATIONS:
        q=oof[(oof.task==task)&(oof.representation==rep)].copy();counts=q.recording_id.value_counts()
        got=q.set_index('recording_id').outer_fold.astype(int)
        checks.append({'task':task,'representation':rep,'expected_n':len(expected),'observed_n':len(q),
          'each_recording_once':bool((counts==1).all()),'complete_ids':set(q.recording_id)==expected,
          'canonical_fold_match':got.sort_index().equals(expected_fold.loc[got.index].sort_index()),
          'probabilities_finite':bool(np.isfinite(q[probability_cols].to_numpy(float)).all()),
          'probabilities_sum_to_one':bool(np.allclose(q[probability_cols].to_numpy(float).sum(1),1,atol=1e-6))})
integrity=pd.DataFrame(checks);integrity.to_csv(TABLES/'07_INTEGRITY_CHECKS.csv',index=False);display(integrity)
bool_cols=['each_recording_once','complete_ids','canonical_fold_match','probabilities_finite','probabilities_sum_to_one']
if not integrity[bool_cols].all().all():raise AssertionError('P1-E16.7F four-endpoint integrity failure.')
print('P1-E16.7F FOUR-ENDPOINT CANONICAL OOF INTEGRITY PASSED')


In [ ]:
# 9. Pooled metrics, resume-safe recording bootstrap CIs, and paired age sensitivity
def pooled_metrics(q,classes):
    pc=[f'probability_{c}' for c in classes]
    return metrics(q.true_label.to_numpy(),q.predicted_label.to_numpy(),q[pc].to_numpy(float),classes)

def safe_name(value): return str(value).lower().replace(' ','_').replace('/','_')
def bootstrap_metric_table(q,classes,path,seed):
    q=q.reset_index(drop=True); strata={c:np.flatnonzero(q.true_label.to_numpy()==c) for c in classes}
    boot=pd.read_csv(path) if path.is_file() else pd.DataFrame()
    rows=[]
    for repetition in range(len(boot),N_BOOT):
        rng=np.random.default_rng(int(seed)+repetition)
        ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes])
        rows.append({'bootstrap_repetition':repetition,**pooled_metrics(q.iloc[ix],classes)})
        if len(rows)>=100 or repetition==N_BOOT-1:
            boot=pd.concat([boot,pd.DataFrame(rows)],ignore_index=True); rows=[]; boot.to_csv(path,index=False)
    return boot

summary_rows=[]
groups=sorted(oof[['task','representation']].drop_duplicates().itertuples(index=False,name=None))
for number,(task,rep) in enumerate(groups,1):
    q=oof.loc[(oof.task==task)&(oof.representation==rep)].reset_index(drop=True)
    classes=TASKS[task]; obs=pooled_metrics(q,classes)
    path=CACHE/f'bootstrap_summary__{task}__{rep}.csv'; b=bootstrap_metric_table(q,classes,path,BASE_SEED+100000*number)
    row={'task':task,'representation':rep,'n_recordings':len(q),**obs,
         'fold_sd':fold_results[(fold_results.task==task)&(fold_results.representation==rep)].balanced_accuracy.std(ddof=1)}
    for m in ['balanced_accuracy','macro_f1','mcc','roc_auc']:
        row[f'{m}_ci_low']=b[m].quantile(.025); row[f'{m}_ci_high']=b[m].quantile(.975)
    summary_rows.append(row); print('summary bootstrap',number,'/',len(groups),task,rep)
summary=pd.DataFrame(summary_rows).sort_values(['task','balanced_accuracy'],ascending=[True,False])
summary.to_csv(TABLES/'08_PERFORMANCE_SUMMARY_WITH_95CI.csv',index=False); display(summary.round(4))

comparisons=[('hybrid_raw','age_only'),('hybrid_raw_plus_age','hybrid_raw'),
             ('regional_raw_plus_age','regional_raw'),('regional_age_normative','regional_raw'),
             ('regional_combined','regional_raw')]
delta_rows=[]
for task_number,(task,classes) in enumerate(TASKS.items(),1):
    for comparison_number,(method,reference) in enumerate(comparisons,1):
        a=oof[(oof.task==task)&(oof.representation==method)].sort_values('recording_id').reset_index(drop=True)
        b=oof[(oof.task==task)&(oof.representation==reference)].sort_values('recording_id').reset_index(drop=True)
        if not np.array_equal(a.recording_id,b.recording_id): raise AssertionError('Paired comparison IDs differ.')
        observed=pooled_metrics(a,classes)['balanced_accuracy']-pooled_metrics(b,classes)['balanced_accuracy']
        strata={c:np.flatnonzero(a.true_label.to_numpy()==c) for c in classes}
        path=CACHE/f'bootstrap_delta__{task}__{method}__vs__{reference}.csv'
        d=pd.read_csv(path) if path.is_file() else pd.DataFrame(columns=['bootstrap_repetition','delta_ba'])
        rows=[]
        for repetition in range(len(d),N_BOOT):
            rng=np.random.default_rng(BASE_SEED+900000+task_number*100000+comparison_number*10000+repetition)
            ix=np.concatenate([rng.choice(strata[c],len(strata[c]),replace=True) for c in classes])
            delta=pooled_metrics(a.iloc[ix],classes)['balanced_accuracy']-pooled_metrics(b.iloc[ix],classes)['balanced_accuracy']
            rows.append({'bootstrap_repetition':repetition,'delta_ba':delta})
            if len(rows)>=100 or repetition==N_BOOT-1:
                d=pd.concat([d,pd.DataFrame(rows)],ignore_index=True); rows=[]; d.to_csv(path,index=False)
        values=d.delta_ba.to_numpy(float); lo,hi=np.quantile(values,[.025,.975])
        p=min(1.0,2*min((np.sum(values<=0)+1)/(len(values)+1),(np.sum(values>=0)+1)/(len(values)+1)))
        delta_rows.append({'task':task,'method':method,'reference':reference,'delta_balanced_accuracy':observed,
                           'ci_low':lo,'ci_high':hi,'ci_excludes_zero':bool(lo>0 or hi<0),'unadjusted_p_value':p})
deltas=pd.DataFrame(delta_rows)

def holm_adjust(values):
    p=np.asarray(values,float); order=np.argsort(p); adjusted=np.empty(len(p)); running=0.0
    for rank,index in enumerate(order):
        running=max(running,(len(p)-rank)*p[index]); adjusted[index]=min(1.0,running)
    return adjusted
deltas['holm_adjusted_p_value']=holm_adjust(deltas.unadjusted_p_value)
deltas['holm_significant']=deltas.holm_adjusted_p_value<.05
deltas.to_csv(TABLES/'09_PAIRED_AGE_SENSITIVITY_DELTAS.csv',index=False); display(deltas.round(4))


In [ ]:
# 10. Age distributions and simple confounding diagnostics
age_by_label=(pd.DataFrame({'label':labels,'age':ages}).groupby('label').age.agg(['count','mean','std','median','min','max']).reset_index())
age_by_label.to_csv(TABLES/'10_AGE_DISTRIBUTION_BY_LABEL.csv',index=False);display(age_by_label.round(2))


In [ ]:
# 11. Publication-ready age-sensitivity figures
sns.set_theme(style='whitegrid',context='paper')
order=list(TASKS)
fig,axes=plt.subplots(2,2,figsize=(13,9),sharey=True); axes=axes.ravel()
for ax,task in zip(axes,order):
    q=summary[summary.task==task].set_index('representation').loc[REPRESENTATIONS].reset_index()
    x=np.arange(len(q)); y=q.balanced_accuracy.to_numpy(); lo=q.balanced_accuracy_ci_low.to_numpy(); hi=q.balanced_accuracy_ci_high.to_numpy()
    ax.errorbar(x,y,yerr=[y-lo,hi-y],fmt='o-',capsize=3,lw=1.5,color='#1F4E79')
    ax.axhline(1/len(TASKS[task]),ls='--',color='#B22222',label='Chance')
    ax.set_xticks(x,[r.replace('_','\n') for r in q.representation],rotation=20,ha='right')
    ax.set_title(task.replace('_',' ')); ax.set_ylim(.25,1.02); ax.set_ylabel('Balanced accuracy (95% CI)')
fig.suptitle('Four-endpoint age sensitivity under identical final outer folds',fontweight='bold')
fig.tight_layout(); fig.savefig(FIGURES/'Figure_01_Four_Endpoint_Age_Sensitivity.pdf',bbox_inches='tight')
fig.savefig(FIGURES/'Figure_01_Four_Endpoint_Age_Sensitivity.png',dpi=600,bbox_inches='tight'); plt.show()

fig,axes=plt.subplots(2,2,figsize=(12,8),sharex=True); axes=axes.ravel()
for ax,task in zip(axes,order):
    q=deltas[deltas.task==task].reset_index(drop=True); y=np.arange(len(q)); x=q.delta_balanced_accuracy.to_numpy()
    ax.errorbar(x,y,xerr=[x-q.ci_low.to_numpy(),q.ci_high.to_numpy()-x],fmt='o',capsize=3,color='#2A9D8F')
    ax.axvline(0,color='black',lw=1); ax.set_yticks(y,[m.replace('_',' ')+' vs '+r.replace('_',' ') for m,r in zip(q.method,q.reference)],fontsize=8)
    ax.set_title(task.replace('_',' ')); ax.set_xlabel('Paired BA difference (95% CI)')
fig.suptitle('Incremental age and age-normalization effects',fontweight='bold')
fig.tight_layout(); fig.savefig(FIGURES/'Figure_02_Paired_Age_Sensitivity_Deltas.pdf',bbox_inches='tight')
fig.savefig(FIGURES/'Figure_02_Paired_Age_Sensitivity_Deltas.png',dpi=600,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(figsize=(8,4.8))
age_plot=age_audit.sort_values('age_mean')
ax.errorbar(age_plot.label,age_plot.age_mean,yerr=age_plot.age_sd,fmt='o',capsize=5,color='#4C78A8')
ax.set_ylabel('Age, years (mean ± SD)'); ax.set_xlabel('Diagnostic group'); ax.set_title('Age distribution in the final development cohort')
fig.tight_layout(); fig.savefig(FIGURES/'Figure_03_Age_Distribution.pdf',bbox_inches='tight')
fig.savefig(FIGURES/'Figure_03_Age_Distribution.png',dpi=600,bbox_inches='tight'); plt.show()


In [ ]:
# 12. Best-representation confusion matrices
best=summary.loc[summary.groupby('task').balanced_accuracy.idxmax(),['task','representation']]
fig,axes=plt.subplots(2,2,figsize=(10,8)); axes=axes.ravel()
for ax,row in zip(axes,best.itertuples(index=False)):
    classes=TASKS[row.task];q=oof[(oof.task==row.task)&(oof.representation==row.representation)];mat=confusion_matrix(q.true_label,q.predicted_label,labels=classes)
    sns.heatmap(mat,annot=True,fmt='d',cmap='Blues',cbar=False,ax=ax,xticklabels=classes,yticklabels=classes);ax.set(title=f'{row.task}: {row.representation}',xlabel='Predicted',ylabel='True')
fig.tight_layout();fig.savefig(FIGURES/'Figure_04_Best_Representation_Confusion_Matrices.png',dpi=600,bbox_inches='tight');plt.show()


## Interpretation guardrails

1. Age-only performance quantifies confounding risk; it does not show that EEG is unnecessary.
2. EEG-only versus age-only estimates whether EEG adds discriminatory information beyond age under identical held-out recordings.
3. EEG-plus-age versus EEG-only estimates incremental predictive contribution from age.
4. Age-normative EEG tests robustness after removing expected age trends using training-fold Normal references only.
5. A reduction after age normalization can reflect removal of confounding, disease-related age interaction, or normative-model uncertainty; it is not automatically model failure.
6. These are internal development sensitivity analyses. They do not replace the frozen official-test or external evaluations.
7. Findings must be reported for all four endpoints, including weak or null Normal-versus-MCI results.


In [ ]:
# 13. Final decision and reproducibility lock
best_rows=summary.loc[summary.groupby('task').balanced_accuracy.idxmax()].copy();best_rows.to_csv(TABLES/'11_BEST_REPRESENTATION_PER_ENDPOINT.csv',index=False)
final_checks=pd.DataFrame([
 {'check':'official test excluded','pass':bool(not dev.official_split.astype(str).str.lower().eq('test').any()),'detail':'train and validation only'},
 {'check':'canonical fold coverage exact','pass':bool(fold_contract.exact_coverage.all()),'detail':str(canonical_path)},
 {'check':'all endpoint-representation combinations present','pass':len(summary)==len(TASKS)*len(REPRESENTATIONS),'detail':len(summary)},
 {'check':'OOF integrity passed','pass':bool(integrity[['each_recording_once','complete_ids','canonical_fold_match','probabilities_finite','probabilities_sum_to_one']].all().all()),'detail':len(oof)},
 {'check':'age-only baseline reported','pass':set(summary[summary.representation=='age_only'].task)==set(TASKS),'detail':4},
 {'check':'paired age comparisons reported','pass':len(deltas)==len(TASKS)*5,'detail':len(deltas)},
 {'check':'Holm-adjusted paired inference complete','pass':deltas.holm_adjusted_p_value.notna().all(),'detail':len(deltas)},
 {'check':'four endpoints include Normal versus MCI','pass':'normal_vs_mci' in set(summary.task),'detail':sorted(summary.task.unique())}])
final_checks.to_csv(TABLES/'12_FINAL_INTEGRITY_CHECKS.csv',index=False);display(final_checks)
if not final_checks['pass'].all():raise AssertionError('P1-E16.7F final integrity failure.')
manifest=[]
for p in sorted(OUTPUT_ROOT.rglob('*')):
    if p.is_file() and p.name!='13_OUTPUT_MANIFEST.csv':manifest.append({'relative_path':str(p.relative_to(OUTPUT_ROOT)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(TABLES/'13_OUTPUT_MANIFEST.csv',index=False)
decision={'experiment':'P1-E16.7F','status':'P1-E16.7F PASSED — FOUR-ENDPOINT AGE AUDIT LOCKED','completed_utc':datetime.now(timezone.utc).isoformat(),
 'canonical_fold_source':str(canonical_path),'canonical_fold_sha256':sha256_file(canonical_path),'official_test_accessed':False,'external_validation_performed':False,
 'endpoints':TASKS,'representations':REPRESENTATIONS,'best_results':best_rows[['task','representation','balanced_accuracy','balanced_accuracy_ci_low','balanced_accuracy_ci_high']].to_dict('records'),
 'next_step':'Integrate the four-endpoint age audit into the final manuscript evidence lock and reviewer response.'}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(decision,indent=2),encoding='utf-8');display(pd.Series(decision,name='value').to_frame());print(decision['status'])
